# Phân tích các yếu tố ảnh hưởng đến rating người dùng trên MovieLens bằng cách kết hợp dữ liệu thông tin phim (TMDb)

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import ast


## Chuẩn bị và Làm sạch Dữ liệu (Data Preparation)

In [57]:
# --- 1. ĐỌC BỘ DỮ LIỆU MOVIELENS 1M ---
# Định nghĩa tên cột cho các file .dat
u_cols = ['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code']
m_cols = ['MovieID', 'Title', 'Genres']
r_cols = ['UserID', 'MovieID', 'Rating', 'Timestamp']

# Đọc file users.dat
users = pd.read_csv('datasets/movie_len_1M/users.dat', sep='::', names=u_cols, encoding='latin-1', engine='python')

# Đọc file movies.dat
movies = pd.read_csv('datasets/movie_len_1M/movies.dat', sep='::', names=m_cols, encoding='latin-1', engine='python')

# Đọc file ratings.dat
ratings = pd.read_csv('datasets/movie_len_1M/ratings.dat', sep='::', names=r_cols, encoding='latin-1', engine='python')

print("--- Đã tải xong MovieLens 1M ---")
print(f"Users shape: {users.shape}")
print(f"Movies shape: {movies.shape}")
print(f"Ratings shape: {ratings.shape}")
print("-" * 30)

--- Đã tải xong MovieLens 1M ---
Users shape: (6040, 5)
Movies shape: (3883, 3)
Ratings shape: (1000209, 4)
------------------------------


In [58]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   UserID      6040 non-null   int64 
 1   Gender      6040 non-null   object
 2   Age         6040 non-null   int64 
 3   Occupation  6040 non-null   int64 
 4   Zip-code    6040 non-null   object
dtypes: int64(3), object(2)
memory usage: 236.1+ KB


In [59]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   MovieID  3883 non-null   int64 
 1   Title    3883 non-null   object
 2   Genres   3883 non-null   object
dtypes: int64(1), object(2)
memory usage: 91.1+ KB


In [60]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   UserID     1000209 non-null  int64
 1   MovieID    1000209 non-null  int64
 2   Rating     1000209 non-null  int64
 3   Timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


In [61]:
# --- 2. ĐỌC VÀ GHÉP NỐI NỘI BỘ DỮ LIỆU TMDB ---
try:
    tmdb_credits = pd.read_csv('datasets/tmdb_5000/tmdb_5000_credits.csv', encoding='utf-8')
    tmdb_movies = pd.read_csv('datasets/tmdb_5000/tmdb_5000_movies.csv', encoding='utf-8')
except UnicodeDecodeError:
    tmdb_credits = pd.read_csv('datasets/tmdb_5000/tmdb_5000_credits.csv', encoding='latin-1')
    tmdb_movies = pd.read_csv('datasets/tmdb_5000/tmdb_5000_movies.csv', encoding='latin-1')

print("--- Đã tải xong TMDB 5000 Credits ---")
print(f"TMDB Credits shape: {tmdb_credits.shape}")
print(f"Các cột trong TMDB: {list(tmdb_credits.columns)}")
print("--- Đã tải xong TMDB 5000 Movies ---")
print(f"TMDB Movies shape: {tmdb_movies.shape}")
print(f"Các cột trong TMDB Movies: {list(tmdb_movies.columns)}")

--- Đã tải xong TMDB 5000 Credits ---
TMDB Credits shape: (4803, 4)
Các cột trong TMDB: ['movie_id', 'title', 'cast', 'crew']
--- Đã tải xong TMDB 5000 Movies ---
TMDB Movies shape: (4803, 20)
Các cột trong TMDB Movies: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


In [62]:
# Chúng ta đổi tên cột trong file credits để khớp nhau trước khi merge
tmdb_credits.columns = ['movie_id', 'tittle_credits', 'cast', 'crew']
tmdb_movies = tmdb_movies.rename(columns={'id': 'movie_id'})

# Ghép nội bộ 2 file TMDB dựa trên 'movie_id'
tmdb_full = pd.merge(tmdb_movies, tmdb_credits, on='movie_id')

print("--- Đã ghép nội bộ TMDB ---")
print(f"Kích thước TMDB sau khi gộp: {tmdb_full.shape}")
# Bạn sẽ thấy có thêm cột budget, revenue, popularity, runtime...

--- Đã ghép nội bộ TMDB ---
Kích thước TMDB sau khi gộp: (4803, 23)


In [63]:
tmdb_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   movie_id              4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [64]:
# --- 3. HIỂN THỊ DỮ LIỆU MẪU ĐỂ KIỂM TRA ---

print("\n--- Sample Users ---")
users.head(3)

print("\n--- Sample Movies ---")
movies.head(3)

print("\n--- Sample Ratings ---")
ratings.head(3)

print("\n--- Sample TMDB Credits ---")
tmdb_credits.head(3)

print("\n--- Sample TMDB Movies ---")
tmdb_movies.head(3)


--- Sample Users ---

--- Sample Movies ---

--- Sample Ratings ---

--- Sample TMDB Credits ---

--- Sample TMDB Movies ---


,budget,genres,homepage,movie_id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466


### Chuẩn hóa

In [65]:
# --- 3. CHUẨN HÓA TÊN PHIM ĐỂ NỐI VỚI MOVIELENS ---

def clean_movielens_title(text):
    text = str(text)
    # Bỏ năm (YYYY) ở cuối
    text = re.sub(r'\s*\(\d{4}\)$', '', text) 
    return text.lower().strip()

def clean_tmdb_title(text):
    return str(text).lower().strip()

# Tạo khóa nối 'title_join'
movies['title_join'] = movies['Title'].apply(clean_movielens_title)
tmdb_full['title_join'] = tmdb_full['title'].apply(clean_tmdb_title)

# --- 4. MERGE CUỐI CÙNG (MOVIELENS + TMDB FULL) ---
movies_merged = pd.merge(movies, tmdb_full, on='title_join', how='inner')

print("-" * 30)
print(f"Số lượng phim MovieLens gốc: {movies.shape[0]}")
print(f"Số lượng phim khớp được với TMDB: {movies_merged.shape[0]}")
print("-" * 30)

# Xem các cột mới có giá trị gì
print("Danh sách các cột hiện có trong bảng movies_merged:")
print(list(movies_merged.columns))


------------------------------
Số lượng phim MovieLens gốc: 3883
Số lượng phim khớp được với TMDB: 892
------------------------------
Danh sách các cột hiện có trong bảng movies_merged:
['MovieID', 'Title', 'Genres', 'title_join', 'budget', 'genres', 'homepage', 'movie_id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'tittle_credits', 'cast', 'crew']


### Tạo đặc trưng

In [ ]:
import pandas as pd
import ast

# --- 1. ĐỊNH NGHĨA HÀM XỬ LÝ (GIỮ NGUYÊN) ---
def get_director(text):
    try:
        crew_data = ast.literal_eval(text)
        for crew_member in crew_data:
            if crew_member['job'] == 'Director':
                return crew_member['name']
        return None
    except (ValueError, SyntaxError):
        return None

def get_top_actors(text):
    try:
        cast_data = ast.literal_eval(text)
        return [member['name'] for member in cast_data[:3]]
    except (ValueError, SyntaxError):
        return []

# --- 2. ÁP DỤNG VÀO DATAFRAME (PHIÊN BẢN AN TOÀN) ---
print("Đang trích xuất dữ liệu Đạo diễn và Diễn viên...")

# BƯỚC QUAN TRỌNG: Xóa các cột cũ nếu đã tồn tại để tránh bị trùng lặp khi chạy lại code
cols_to_reset = ['director', 'actor_1', 'actor_2', 'actor_3', 'actors_list']
movies_merged = movies_merged.drop(columns=[c for c in cols_to_reset if c in movies_merged.columns], errors='ignore')

# 1. Trích xuất Đạo diễn
movies_merged['director'] = movies_merged['crew'].apply(get_director)

# 2. Trích xuất List Diễn viên tạm thời
movies_merged['actors_list'] = movies_merged['cast'].apply(get_top_actors)

# 3. Tách List thành 3 cột riêng biệt
# apply(pd.Series) sẽ biến list [A, B, C] thành 3 cột 0, 1, 2
actors_df = movies_merged['actors_list'].apply(pd.Series)

# Đảm bảo luôn có đủ 3 cột (xử lý trường hợp dữ liệu ít diễn viên)
for i in range(3):
    if i not in actors_df.columns:
        actors_df[i] = None

# Đổi tên cột chuẩn xác
actors_df = actors_df.rename(columns={0: 'actor_1', 1: 'actor_2', 2: 'actor_3'})

# 4. Ghép vào bảng chính
# Chỉ lấy 3 cột đầu tiên (đề phòng trường hợp hiếm hoi có cột thừa)
movies_merged = pd.concat([movies_merged, actors_df[['actor_1', 'actor_2', 'actor_3']]], axis=1)

# 5. Dọn dẹp: Xóa cột list tạm và các cột json gốc cho nhẹ máy
# (Tùy chọn: Bạn có thể giữ lại cast/crew nếu muốn, nhưng ở đây tôi xóa cho gọn như yêu cầu trước)
movies_merged = movies_merged.drop(columns=['actors_list'], errors='ignore')

# --- 3. KIỂM TRA KẾT QUẢ ---
print("-" * 30)
print("Các cột hiện có sau khi xử lý:")
print(list(movies_merged.columns))

print("\n5 dòng dữ liệu mẫu:")
print(movies_merged[['Title', 'director', 'actor_1', 'actor_2', 'actor_3']].head())

Đang trích xuất dữ liệu Đạo diễn và Diễn viên...
------------------------------
                          Title            director             actor_1  \
538               Psycho (1998)    Alfred Hitchcock     Anthony Perkins   
484          Family Plot (1976)    Alfred Hitchcock      Barbara Harris   
119         True Romance (1993)          Tony Scott    Christian Slater   
325        Event Horizon (1997)  Paul W.S. Anderson  Laurence Fishburne   
567  Blast from the Past (1999)         Hugh Wilson      Brendan Fraser   

                actor_1             actor_2             actor_2  \
538     Anthony Perkins          Vera Miles          Vera Miles   
484      Barbara Harris          Bruce Dern          Bruce Dern   
119    Christian Slater   Patricia Arquette   Patricia Arquette   
325  Laurence Fishburne           Sam Neill           Sam Neill   
567      Brendan Fraser  Alicia Silverstone  Alicia Silverstone   

                actor_3             actor_3  
538          John Ga

In [70]:
# --- TỐI ƯU HÓA TRƯỚC KHI MERGE ---

# 1. Chọn lọc các cột cần thiết từ bảng phim
# Chỉ lấy những gì bạn thực sự định phân tích
cols_to_keep = [
    'MovieID', 'Title', 'title',       # Định danh
    'director', 'actor_1', 'actor_2', 'actor_3', # Con người
    'budget', 'revenue', 'vote_average', 'runtime', # Tài chính/Chất lượng
    'Genres', 'year' # Nếu bạn có trích xuất năm
]

# Lọc bớt cột không tồn tại để tránh lỗi (dùng list comprehension)
existing_cols = [c for c in cols_to_keep if c in movies_merged.columns]
movies_lean = movies_merged[existing_cols]

# 2. Thực hiện Merge
# Inner join là chuẩn xác: Chúng ta chỉ phân tích Rating của những phim ĐÃ CÓ thông tin Director/Actor
full_data = pd.merge(ratings, movies_lean, on='MovieID', how='inner')

# --- KIỂM TRA KẾT QUẢ ---
print(f"Tổng số lượt đánh giá ban đầu: {ratings.shape[0]}")
print(f"Tổng số dòng dữ liệu sau khi merge: {full_data.shape[0]}")
print(f"-> Số lượng đánh giá bị loại bỏ (do phim không khớp TMDB): {ratings.shape[0] - full_data.shape[0]}")

# Xem qua dữ liệu cuối cùng
print("\n5 dòng dữ liệu hoàn chỉnh để phân tích:")
print(full_data[['UserID', 'Rating', 'Title', 'director', 'actor_1']].head())

Tổng số lượt đánh giá ban đầu: 1000209
Tổng số dòng dữ liệu sau khi merge: 450575
-> Số lượng đánh giá bị loại bỏ (do phim không khớp TMDB): 549634

5 dòng dữ liệu hoàn chỉnh để phân tích:
   UserID  Rating                                   Title           director  \
0       1       5  One Flew Over the Cuckoo's Nest (1975)       Miloš Forman   
1       1       3                     My Fair Lady (1964)       George Cukor   
2       1       4                  Erin Brockovich (2000)  Steven Soderbergh   
3       1       5                          Ben-Hur (1959)  Timur Bekmambetov   
4       1       4  Snow White and the Seven Dwarfs (1937)         David Hand   

             actor_1            actor_1  
0     Jack Nicholson     Jack Nicholson  
1     Audrey Hepburn     Audrey Hepburn  
2      Julia Roberts      Julia Roberts  
3        Jack Huston        Jack Huston  
4  Adriana Caselotti  Adriana Caselotti  


In [68]:
full_data.head(10)

,UserID,MovieID,Rating,Timestamp,Title,Genres,title_join,budget,genres,homepage,...,vote_average,vote_count,tittle_credits,cast,crew,director,actors,actor_1,actor_2,actor_3
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama,one flew over the cuckoo's nest,3000000,"[{""id"": 18, ""name"": ""Drama""}]",NaN,...,8.2,2919,One Flew Over the Cuckoo's Nest,"[{""cast_id"": 15, ""character"": ""Randle Patrick ...","[{""credit_id"": ""52fe424cc3a36847f80134e7"", ""de...",Miloš Forman,"[Jack Nicholson, Louise Fletcher, Danny DeVito]",Jack Nicholson,Louise Fletcher,Danny DeVito
1,1,914,3,978301968,My Fair Lady (1964),Musical|Romance,my fair lady,17000000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10751, ""n...",NaN,...,7.4,352,My Fair Lady,"[{""cast_id"": 1, ""character"": ""Eliza Doolittle""...","[{""credit_id"": ""52fe43fb9251416c75024edb"", ""de...",George Cukor,"[Audrey Hepburn, Rex Harrison, Stanley Holloway]",Audrey Hepburn,Rex Harrison,Stanley Holloway
2,1,3408,4,978300275,Erin Brockovich (2000),Drama,erin brockovich,52000000,"[{""id"": 18, ""name"": ""Drama""}]",http://www.brockovich.com/movie.html,...,7.1,716,Erin Brockovich,"[{""cast_id"": 25, ""character"": ""Erin Brockovich...","[{""credit_id"": ""58b999c0c3a36866af00ba1d"", ""de...",Steven Soderbergh,"[Julia Roberts, Albert Finney, Aaron Eckhart]",Julia Roberts,Albert Finney,Aaron Eckhart
3,1,1287,5,978302039,Ben-Hur (1959),Action|Adventure|Drama,ben-hur,100000000,"[{""id"": 18, ""name"": ""Drama""}]",http://www.benhurmovie.com/,...,5.3,621,Ben-Hur,"[{""cast_id"": 1, ""character"": ""Judah Ben-Hur"", ...","[{""credit_id"": ""587811b7925141104a00cb16"", ""de...",Timur Bekmambetov,"[Jack Huston, Toby Kebbell, Rodrigo Santoro]",Jack Huston,Toby Kebbell,Rodrigo Santoro
4,1,594,4,978302268,Snow White and the Seven Dwarfs (1937),Animation|Children's|Musical,snow white and the seven dwarfs,1488423,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 16, ""na...",http://movies.disney.com/snow-white-and-the-se...,...,6.9,1914,Snow White and the Seven Dwarfs,"[{""cast_id"": 14, ""character"": ""Snow White (voi...","[{""credit_id"": ""52fe4240c3a36847f800fa21"", ""de...",David Hand,"[Adriana Caselotti, Lucille La Verne, Harry St...",Adriana Caselotti,Lucille La Verne,Harry Stockwell
5,1,2791,4,978302188,Airplane! (1980),Comedy,airplane!,3500000,"[{""id"": 35, ""name"": ""Comedy""}]",NaN,...,7.1,1074,Airplane!,"[{""cast_id"": 5, ""character"": ""Ted Striker"", ""c...","[{""credit_id"": ""52fe427ac3a36847f8021c35"", ""de...",Jerry Zucker,"[Robert Hays, Julie Hagerty, Kareem Abdul-Jabbar]",Robert Hays,Julie Hagerty,Kareem Abdul-Jabbar
6,1,2018,4,978301777,Bambi (1942),Animation|Children's,bambi,858000,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 18, ""...",http://movies.disney.com/bambi,...,6.8,1405,Bambi,"[{""cast_id"": 9, ""character"": ""Young Bambi (voi...","[{""credit_id"": ""52fe438cc3a36847f805ca73"", ""de...",David Hand,"[Donnie Dunagan, Peter Behn, Stan Alexander]",Donnie Dunagan,Peter Behn,Stan Alexander
7,1,2797,4,978302039,Big (1988),Comedy|Fantasy,big,18000000,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 18, ""na...",NaN,...,6.9,993,Big,"[{""cast_id"": 1, ""character"": ""Joshua \""Josh\"" ...","[{""credit_id"": ""52fe4349c3a36847f8048aef"", ""de...",Penny Marshall,"[Tom Hanks, Elizabeth Perkins, Robert Loggia]",Tom Hanks,Elizabeth Perkins,Robert Loggia
8,1,1270,5,978300055,Back to the Future (1985),Comedy|Sci-Fi,back to the future,19000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 35, ""...",http://www.backtothefuture.com/movies/backtoth...,...,8.0,6079,Back to the Future,"[{""cast_id"": 14, ""character"": ""Marty McFly"", ""...","[{""credit_id"": ""52fe4218c3a36847f80039eb"", ""de...",Robert Zemeckis,"[Michael J. Fox, Christopher Lloyd, Lea Thompson]",Michael J. Fox,Christopher Lloyd,Lea Thompson
9,1,527,5,978824195,Schindler's List (1993),Drama|War,schindler's list,22000000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 36, ""name...",http://www.s

## Khám phá Dữ liệu (EDA - Exploratory Data Analysis)

## Phân tích Chuyên sâu - Các yếu tố ảnh hưởng Rating

+ Tác động của Đạo diễn (The Director's Cut)
+ Sức hút của Diễn viên (Star Power)

### Tác động của Đạo diễn (The Director's Cut)

### Sức hút của Diễn viên (Star Power)

## Kết luận và Trực quan hóa